In [21]:
import uuid
from langgraph.graph import StateGraph, START , END
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage,HumanMessage
from pydantic import Field,BaseModel
from langgraph.graph.message import add_messages
from typing import List,TypedDict,Annotated
from dotenv import load_dotenv

In [20]:
load_dotenv()

True

In [15]:
class Task(BaseModel):
    id : int
    title : str
    brief : str = Field(...,description="What to Cover ?")

In [16]:
class Plan(BaseModel):
    blog_title : str
    tasks : List[Task]

In [17]:
class State(TypedDict):
    topic : str
    plan : Plan
    sections : Annotated[List[str] , add_messages]
    final : str

In [18]:
llm = ChatOpenAI()

In [22]:
def orchestrator(state : State) -> dict :
    # the purpose of this node will be to plan the entire steps that will be executed by workers to finish the task.
    plan = llm.with_structured_output(schema=Plan).invoke(
        [
            SystemMessage(content= "Create a blog plan with 5-7 sections on the following topic."),
            HumanMessage(content=f"Topic : {state['topic']}")
        ]
    )

    return {'plan' : plan}